# Quarterly Flight Summary

This notebook summarizes the analysis-ready flight data by shifted fiscal quarters: Q1 is December through February, Q2 is March through May, Q3 is June through August, and Q4 is September through November. Fiscal years are labeled by their ending year. The cleaned dataset retains flights with a recorded arrival delay, so flight counts and delay statistics describe that analysis sample.

In [4]:
import pandas as pd

raw_flights = pd.read_csv("data/raw/flights.csv")
weather = pd.read_csv("data/raw/weather.csv")
airlines = pd.read_csv("data/raw/airlines.csv")
flights = pd.read_csv("data/cleaned/analysis_ready_flights.csv")

print(f"Raw flights: {raw_flights.shape}")
print(f"Weather: {weather.shape}")
print(f"Airlines: {airlines.shape}")
print(f"Analysis-ready flights: {flights.shape}")

print("\nRaw flights preview:")
display(raw_flights.head())
print("Weather preview:")
display(weather.head())
print("Airlines preview:")
display(airlines.head())

flights["quarter"] = ((flights["month"] % 12) // 3 + 1).astype(int)
flights["fiscal_year"] = (
    flights["year"].astype(int) + (flights["month"] == 12).astype(int)
)
flights["year_quarter"] = (
    "FY"
    + flights["fiscal_year"].astype(str)
    + " Q"
    + flights["quarter"].astype(str)
)

print("\nFiscal quarterly periods:", sorted(flights["year_quarter"].unique()))

Raw flights: (336776, 19)
Weather: (26115, 15)
Airlines: (16, 2)
Analysis-ready flights: (327346, 34)

Raw flights preview:


,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01T10:00:00Z
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01T10:00:00Z
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01T10:00:00Z
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01T10:00:00Z
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01T11:00:00Z


Weather preview:


,origin,year,month,day,hour,temp,dewp,humid,wind_dir,wind_speed,wind_gust,precip,pressure,visib,time_hour
0,EWR,2013,1,1,1,39.02,26.06,59.37,270.0,10.35702,NaN,0.0,1012.0,10.0,2013-01-01T06:00:00Z
1,EWR,2013,1,1,2,39.02,26.96,61.63,250.0,8.05546,NaN,0.0,1012.3,10.0,2013-01-01T07:00:00Z
2,EWR,2013,1,1,3,39.02,28.04,64.43,240.0,11.50780,NaN,0.0,1012.5,10.0,2013-01-01T08:00:00Z
3,EWR,2013,1,1,4,39.92,28.04,62.21,250.0,12.65858,NaN,0.0,1012.2,10.0,2013-01-01T09:00:00Z
4,EWR,2013,1,1,5,39.02,28.04,64.43,260.0,12.65858,NaN,0.0,1011.9,10.0,2013-01-01T10:00:00Z


Airlines preview:


,carrier,name
0,9E,Endeavor Air Inc.
1,AA,American Airlines Inc.
2,AS,Alaska Airlines Inc.
3,B6,JetBlue Airways
4,DL,Delta Air Lines Inc.



Fiscal quarterly periods: ['FY2013 Q1', 'FY2013 Q2', 'FY2013 Q3', 'FY2013 Q4', 'FY2014 Q1']


In [6]:
quarterly_summary = (
    flights.groupby(["fiscal_year", "quarter"], as_index=False)
    .agg(
        flight_count=("arr_delay", "size"),
        mean_dep_delay_min=("dep_delay", "mean"),
        median_dep_delay_min=("dep_delay", "median"),
        mean_arr_delay_min=("arr_delay", "mean"),
        median_arr_delay_min=("arr_delay", "median"),
        pct_arrival_delay_over_15=(
            "arr_delay", lambda delay: delay.gt(15).mean() * 100
        ),
        mean_distance_miles=("distance", "mean"),
        mean_air_time_min=("air_time", "mean"),
        mean_visibility_miles=("visib", "mean"),
        mean_wind_speed_mph=("wind_speed", "mean"),
        mean_temperature_f=("temp", "mean"),
        mean_precipitation_inches=("precip", "mean"),
    )
)

quarterly_summary = quarterly_summary.round(2)
display(quarterly_summary)

assert quarterly_summary["flight_count"].sum() == len(flights)
assert quarterly_summary["quarter"].between(1, 4).all()
assert flights.loc[flights["month"] == 12, "quarter"].eq(1).all()
print("Shifted quarterly summary validation passed.")

,fiscal_year,quarter,flight_count,mean_dep_delay_min,median_dep_delay_min,mean_arr_delay_min,median_arr_delay_min,pct_arrival_delay_over_15,mean_distance_miles,mean_air_time_min,mean_visibility_miles,mean_wind_speed_mph,mean_temperature_f,mean_precipitation_inches
0,2013,1,50009,10.35,-2.0,5.89,-3.0,22.83,1010.70,152.85,8.84,11.89,36.11,0.00
1,2013,2,83594,13.30,-2.0,6.81,-5.0,24.23,1039.76,149.28,9.26,12.01,52.81,0.00
2,2013,3,84124,18.21,0.0,12.99,-3.0,28.64,1071.21,148.38,9.53,10.29,77.18,0.01
3,2013,4,82599,6.10,-3.0,-1.22,-9.0,15.80,1047.49,149.26,9.58,10.67,58.98,0.00
4,2014,1,27020,16.48,0.0,14.87,2.0,32.61,1076.33,162.59,8.56,10.16,39.43,0.01


Shifted quarterly summary validation passed.
